In [31]:
import pandas as pd
import pdfplumber
import os

DATA = "data/"

# ── Column-name map ──────────────────────────────────────────────────────────────
# If your CSV/XLS export uses different headers, only change the values here.
COL = {
    # Bestellungen
    "order_id":        "Bestellnummer",
    "artikel":         "Artikel",
    "preis":           "Preis",
    "email":           "E-Mail-Adresse des Kontakts",
    "erstelldatum":    "Erstellungsdatum",
    "rech_adresse":    "Adresse der Rechnung",
    "rech_ort":        "Ort der Rechnung",
    "rech_plz":        "Rechnungs-PLZ",
    "rech_land":       "Land der Rechnung",
    "rech_bundesland": "Bundesland der Rechnung",
    "rech_unternehmen":"Unternehmen",
    # Kontakte
    "kontakt_email":   "E-Mail-Adresse 1",
    "titel":           "Titel",
    "vorname":         "Vorname",
    "nachname":        "Nachname",
    "telefon":         "Telefonnummer 1",
    "geburtstag":      "Geburtsdatum",
    "gender":          "Gender",
}

# ── Helpers ───────────────────────────────────────────────────────────────────────

def load_csv(path, **kwargs):
    """Load a semicolon-separated CSV; on encoding errors, fall back to utf-8 then latin-1."""
    encodings = list(dict.fromkeys([kwargs.pop("encoding", "utf-8"), "utf-8", "latin-1"]))
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, sep=";", encoding=enc, **kwargs)
        except (UnicodeDecodeError, LookupError) as e:
            last_err = e
    raise ValueError(f"Could not read '{path}' with encodings {encodings}: {last_err}")

def get_col(df, col, default=""):
    """Return df[col] if it exists, otherwise an empty Series with a warning."""
    if col in df.columns:
        return df[col]
    print(f"  ⚠  Column '{col}' not found – filling with {default!r}")
    return pd.Series(default, index=df.index, dtype=str)

def align_columns(source_df, target_cols, label=""):
    """Return source_df restricted to target_cols that actually exist; warn about gaps."""
    present = [c for c in target_cols if c in source_df.columns]
    missing = [c for c in target_cols if c not in source_df.columns]
    if missing:
        print(f"  ⚠  {label}: columns missing from new_members – skipping: {missing}")
    return source_df[present].copy()

def dedup_append(existing: pd.DataFrame, new_rows: pd.DataFrame, label="") -> pd.DataFrame:
    """
    Append new_rows to existing, skipping any row whose dedup key already appears.
    Dedup key: 'Email' if present, else 'Membership Number #'.
    Rows with a null/empty key are never considered duplicates (always kept).
    Existing rows always win (keep='first').
    """
    for key in ("Email", "Membership Number #"):
        if key in existing.columns and key in new_rows.columns:
            break
    else:
        # No usable key – fall back to plain concat
        print(f"  ⚠  {label}: no dedup key found, appending without dedup")
        return pd.concat([existing, new_rows], ignore_index=True)

    combined = pd.concat([existing, new_rows], ignore_index=True)

    # Only deduplicate rows that have a non-null, non-empty key value
    has_key  = combined[key].notna() & (combined[key].astype(str).str.strip() != "")
    deduped  = combined[has_key].drop_duplicates(subset=[key], keep="first")
    no_key   = combined[~has_key]
    result   = pd.concat([deduped, no_key]).sort_index().reset_index(drop=True)

    skipped = len(combined) - len(result)
    if skipped:
        print(f"  ℹ  {label}: {skipped} duplicate(s) skipped (matched on '{key}')")
    return result

# ── Load data ─────────────────────────────────────────────────────────────────────
bestellungen = load_csv(DATA + "Bestellungen von strich.jan.csv",  encoding="ascii")
kontakte     = load_csv(DATA + "Kontakte von strich.jan.csv",      encoding="windows-1252")
ipna_ml      = load_csv(DATA + "IPNA Masterliste.csv",             encoding="windows-1252")
neue_ml      = load_csv(DATA + "Masterliste neue Mitglieder.csv",  encoding="windows-1250")
voll_ml      = load_csv(DATA + "Masterliste_full.csv",             encoding="windows-1252")

# Inspect PDF structure
with pdfplumber.open(DATA + "beitrage_2026.pdf") as pdf:
    for i, page in enumerate(pdf.pages[:2]):
        print(f"\n=== PDF Page {i+1} ===")
        tables = page.extract_tables()
        if tables:
            for t in tables:
                for row in t[:5]:
                    print(row)
        else:
            print(page.extract_text()[:500])



=== PDF Page 1 ===
['ESPN & IPNA', 'IPNA', 'Membership', 'Note']
['140 €', '0 €', 'ESPN', '']
['192 €', '52 €', 'ESPN&IPNA online journal', '']
['248 €', '108 €', 'ESPN&IPNA online&print journal', '']
['100 €', '0 €', 'ESPN lower-middle income', '']


In [32]:
# --- Block 1: Parse PDF fee table + merge Bestellungen & Kontakte → new_members ---

# 1a. Collect all rows from all PDF tables
with pdfplumber.open(DATA + "beitrage_2026.pdf") as pdf:
    all_rows = [row for page in pdf.pages for table in page.extract_tables() for row in table]

if not all_rows:
    raise ValueError("No tables found in the PDF – check the file path and format.")

# Detect header: first row where at least one cell is non-empty
header_idx = next(
    (i for i, r in enumerate(all_rows) if any(c and str(c).strip() for c in r)), 0
)
raw_header = [str(c).strip() if c else f"col_{i}" for i, c in enumerate(all_rows[header_idx])]
fee_df = pd.DataFrame(all_rows[header_idx + 1:], columns=raw_header)
fee_df = fee_df[fee_df.iloc[:, 0].notna() & (fee_df.iloc[:, 0].str.strip() != "")]

# Auto-detect relevant columns by substring (case-insensitive)
def find_col(df, *keywords, exclude=()):
    for col in df.columns:
        cl = col.lower()
        if all(k.lower() in cl for k in keywords) and not any(e.lower() in cl for e in exclude):
            return col
    return None

mem_col       = find_col(fee_df, "membership") or fee_df.columns[0]
espn_ipna_col = find_col(fee_df, "espn", "ipna") or find_col(fee_df, "espn") or fee_df.columns[1]
ipna_amt_col  = find_col(fee_df, "ipna", exclude=("espn",)) or fee_df.columns[2]

fee_df = fee_df.drop_duplicates(subset=mem_col, keep="first")
fee_lookup = fee_df.set_index(mem_col)[[espn_ipna_col, ipna_amt_col]].to_dict("index")

print(f"PDF columns detected: membership='{mem_col}', espn_ipna='{espn_ipna_col}', ipna='{ipna_amt_col}'")
print("Fee schedule loaded:", list(fee_lookup.keys()))

# 1b. Map "Artikel" string → normalised Membership label
# Keywords are matched case-insensitively; adjust strings here if article names change.
def parse_membership(artikel):
    a = str(artikel).lower() if pd.notna(artikel) else ""
    base = "ESPN&IPNA" if ("espn" in a and "ipna" in a) else ("ESPN" if "espn" in a else "IPNA")
    lmi  = "lower-middle income -" if "lower" in a else ""
    jrn  = "online&print journal" if "print" in a else ("online journal" if "online" in a else "")
    return " ".join(filter(None, [base, lmi, jrn]))

# 1c. Validate merge keys then join on e-mail
email_l, email_r = COL["email"], COL["kontakt_email"]
for col, df, name in [(email_l, bestellungen, "Bestellungen"), (email_r, kontakte, "Kontakte")]:
    if col not in df.columns:
        raise KeyError(f"Email column '{col}' not found in {name}. Available: {df.columns.tolist()}")

merged = bestellungen.merge(kontakte, left_on=email_l, right_on=email_r, how="left")
merged["Membership"] = merged[COL["artikel"]].apply(parse_membership)

# 1d. Format numeric amount: "152.00" / "152,00" / "152 €" → "152,00 €"
#     Strips any trailing € and surrounding whitespace before converting.
def fmt_amount(val):
    try:
        cleaned = str(val).replace("€", "").strip().replace(",", ".")
        return f"{float(cleaned):.2f} €".replace(".", ",")
    except (ValueError, TypeError):
        return str(val)

# 1e. IPNA amount from PDF lookup, normalised to "X,XX €" format (falls back to "0,00 €")
def get_ipna_amount(membership):
    raw = fee_lookup.get(membership, {}).get(ipna_amt_col, "0 €")
    return fmt_amount(raw)

# 1f. Assemble new_members; missing source columns produce empty strings + a warning
new_members = pd.DataFrame({
    "Membership Number #": get_col(merged, COL["order_id"]),
    "Titel":               get_col(merged, COL["titel"]),
    "First Name":          get_col(merged, COL["vorname"]),
    "Last Name":           get_col(merged, COL["nachname"]),
    "Email":               get_col(merged, COL["email"]),
    "Phone":               get_col(merged, COL["telefon"]),
    "Birthdate":           get_col(merged, COL["geburtstag"]),
    "Address":             get_col(merged, COL["rech_adresse"]),
    "City":                get_col(merged, COL["rech_ort"]),
    "Zipcode":             get_col(merged, COL["rech_plz"]),
    "Country":             get_col(merged, COL["rech_land"]),
    "State":               get_col(merged, COL["rech_bundesland"]),
    "Company":             get_col(merged, COL["rech_unternehmen"]),
    "Member since":        get_col(merged, COL["erstelldatum"]),
    "ESPN&IPNA amount":    get_col(merged, COL["preis"]).apply(fmt_amount),
    "IPNA amount":         merged["Membership"].apply(get_ipna_amount),
    "Membership":          merged["Membership"],
    "Gender":              get_col(merged, COL["gender"]),
    "Note":                "",
})

print(f"\nNew members to add: {len(new_members)}")
display(new_members)

# 1g. Update & save IPNA Masterliste (only columns present in both)
new_for_ipna = align_columns(new_members, ipna_ml.columns.tolist(), label="IPNA Masterliste")
updated_ipna = dedup_append(ipna_ml, new_for_ipna, label="IPNA Masterliste")
updated_ipna.to_csv(DATA + "IPNA Masterliste.csv", sep=";", index=False, encoding="windows-1252")

print(f"\nIPNA Masterliste: {len(ipna_ml)} → {len(updated_ipna)} rows")
display(updated_ipna.tail(len(new_for_ipna) + 2))


PDF columns detected: membership='Membership', espn_ipna='ESPN & IPNA', ipna='IPNA'
Fee schedule loaded: ['ESPN', 'ESPN&IPNA online journal', 'ESPN&IPNA online&print journal', 'ESPN lower-middle income', 'ESPN&IPNA lower-middle income - online journal', 'ESPN&IPNA lower-middle income - online&print journal', 'ESPN low income', 'ESPN&IPNA low income - online journal', 'ESPN&IPNA low income - online&print journal', 'ESPN Membership 2026 Honorary Members', 'ESPN / ESPN lower-middle income / ESPN low income']

New members to add: 1


,Membership Number #,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,IPNA amount,Membership,Gender,Note
0,11929,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €","52,00 €",ESPN&IPNA lower-middle income - online journal,NaN,


  ℹ  IPNA Masterliste: 1 duplicate(s) skipped (matched on 'Email')

IPNA Masterliste: 3 → 3 rows


,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,IPNA amount,Membership,Gender,Note
0,Associate Professor,IULIANA MAGDALENA,STARCEA,magdabirm@yahoo.com,'+40 726 704 612,10.07.1974,"62-64, V. Lupu street",Ia?i,700648,Romania,Ia?i,"Pediatric Nephrology Department, Grigore T. Po...",10.01.2024 17:47,"152,00 Û","52,00 Û",ESPN&IPNA lower-middle income - online journal,Female,NaN
1,Dr.,Bohuslav,Gibl‡k,giblakbohuslav@gmail.com,'+421 944 364 707,13.06.1989,Meden‡ 10,Bansk‡ Bystrica,97405,Slovakia,Banska Bystrica,Children hospital - Detsk‡ fakultn‡ nemocnica ...,15.05.2024 19:39,"152,00 Û","52,00 Û",ESPN&IPNA lower-middle income - online journal,Male,NaN
2,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €",52 €,ESPN&IPNA lower-middle income - online journal,NaN,NaN


In [33]:
# --- Block 2: Update Masterliste neue Mitglieder ---

new_for_neue = align_columns(new_members, neue_ml.columns.tolist(), label="Masterliste neue Mitglieder")
updated_neue = dedup_append(neue_ml, new_for_neue, label="Masterliste neue Mitglieder")
updated_neue.to_csv(DATA + "Masterliste neue Mitglieder.csv", sep=";", index=False, encoding="windows-1250")

print(f"Masterliste neue Mitglieder: {len(neue_ml)} → {len(updated_neue)} rows")
display(updated_neue.tail(len(new_for_neue) + 2))


  ℹ  Masterliste neue Mitglieder: 1 duplicate(s) skipped (matched on 'Email')
Masterliste neue Mitglieder: 3 → 3 rows


,Membership Number #,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,Membership,Gender,Note
0,11924,Assoc Prof,Serra,Sźrmeli Dšven,serrasurmel@yahoo.com,'+90 505 919 98 86,17.11.1982,?smet ?nšnź Bulvarő,Yeni?ehir,33120,Turkey,Mersin,Mersin University Faculty of Medicine,27.05.2024 06:08,"140,00 Ű",ESPN,Female,NaN
1,11925,Dr.,Manuela,Colucci,manuela.colucci@opbg.net,'+39 320 760 7919,04.12.1980,"Via Luigi Canina, 6",Roma,196,Italy,Lazio,Bambino Gesť Children Hospital (IRCCS),25.03.2024 16:05,"140,00 Ű",ESPN,Female,NaN
2,11929,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €",ESPN&IPNA lower-middle income - online journal,NaN,NaN


In [34]:
# --- Block 3: Update Masterliste Vollständige Liste ---

new_for_voll = align_columns(new_members, voll_ml.columns.tolist(), label="Masterliste Vollständige Liste")
updated_voll = dedup_append(voll_ml, new_for_voll, label="Masterliste Vollständige Liste")
updated_voll.to_csv(DATA + "Masterliste_full.csv", sep=";", index=False, encoding="windows-1252")

print(f"Masterliste Vollständige Liste: {len(voll_ml)} → {len(updated_voll)} rows")
display(updated_voll.tail(len(new_for_voll) + 2))


  ℹ  Masterliste Vollständige Liste: 1 duplicate(s) skipped (matched on 'Email')
Masterliste Vollständige Liste: 4 → 4 rows


,Membership Number #,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,Membership,Gender,Note
1,11928.0,Dr.,Rik,Westland,ri.westland@amsterdamumc.nl,'+31 6 50063988,08.11.1985,Meibergdreef 9,Amsterdam,1105 AZ,Netherlands,Noord-Holland,Amsterdam UMC,02.01.2024 18:52,"192,00 Û",ESPN&IPNA online journal,Male,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11929.0,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €",ESPN&IPNA lower-middle income - online journal,NaN,NaN


In [24]:
# --- Block 4: Format comparison – new_members vs. original masterliste ---

import re

def compare_format(original: pd.DataFrame, appended: pd.DataFrame, label: str):
    """
    Compare the appended rows (new_members slice) against the schema of the original.
    Checks: column set, column order, dtypes, and per-column value patterns.
    """
    orig_cols  = original.columns.tolist()
    new_rows   = appended.iloc[len(original):]   # just the appended slice

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")

    # 1. Column presence
    missing_in_new  = [c for c in orig_cols if c not in appended.columns]
    extra_in_new    = [c for c in appended.columns if c not in orig_cols]
    order_ok        = appended.columns.tolist() == orig_cols
    print(f"  Columns missing : {missing_in_new or '—'}")
    print(f"  Extra columns   : {extra_in_new   or '—'}")
    print(f"  Column order OK : {order_ok}")

    # 2. Per-column dtype + value-pattern comparison
    def sample_pattern(series):
        """Return a regex that summarises non-null values (numeric / date / euro / text)."""
        s = series.dropna().astype(str).str.strip()
        s = s[s != ""]
        if s.empty:
            return "(empty)"
        if s.str.match(r"^\d{1,3}(,\d{3})*(\.\d+)? €$").mean() > 0.5:
            return "euro-amount"
        if s.str.match(r"^\d{4}-\d{2}-\d{2}").mean() > 0.5:
            return "date (YYYY-MM-DD)"
        if s.str.match(r"^\d+$").mean() > 0.7:
            return "numeric string"
        return "free text"

    print(f"\n  {'Column':<30} {'orig dtype':<12} {'new dtype':<12} {'orig pattern':<22} {'new pattern'}")
    print(f"  {'-'*105}")
    for col in orig_cols:
        if col not in appended.columns:
            continue
        orig_pat = sample_pattern(original[col])
        new_pat  = sample_pattern(new_rows[col]) if not new_rows.empty else "(no new rows)"
        dtype_ok = "✓" if original[col].dtype == appended[col].dtype else "≠"
        pat_ok   = "✓" if orig_pat == new_pat else "≠"
        flag     = "" if (dtype_ok == "✓" and pat_ok == "✓") else "  ← CHECK"
        print(f"  {col:<30} {str(original[col].dtype):<12} {str(appended[col].dtype):<12} {orig_pat:<22} {new_pat}{flag}")

    # 3. Spot-check: sample of appended rows
    if not new_rows.empty:
        print(f"\n  Sample of appended rows ({min(3, len(new_rows))} of {len(new_rows)}):")
        display(new_rows.head(3))
    else:
        print("\n  ⚠  No new rows were appended.")


compare_format(voll_ml,  updated_voll,  "Masterliste_full.csv")
compare_format(neue_ml,  updated_neue,  "Masterliste neue Mitglieder.csv")
compare_format(ipna_ml,  updated_ipna,  "IPNA Masterliste.csv")



  Masterliste_full.csv
  Columns missing : —
  Extra columns   : —
  Column order OK : True

  Column                         orig dtype   new dtype    orig pattern           new pattern
  ---------------------------------------------------------------------------------------------------------
  Membership Number #            float64      float64      free text              free text
  Titel                          str          str          free text              (empty)  ← CHECK
  First Name                     str          str          free text              (empty)  ← CHECK
  Last Name                      str          str          free text              (empty)  ← CHECK
  Email                          str          str          free text              free text
  Phone                          str          str          free text              (empty)  ← CHECK
  Birthdate                      str          str          free text              (empty)  ← CHECK
  Address                

,Membership Number #,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,Membership,Gender,Note
636,11929.0,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €",ESPN&IPNA lower-middle income - online journal,NaN,



  Masterliste neue Mitglieder.csv
  Columns missing : —
  Extra columns   : —
  Column order OK : True

  Column                         orig dtype   new dtype    orig pattern           new pattern
  ---------------------------------------------------------------------------------------------------------
  Membership Number #            int64        int64        numeric string         numeric string
  Titel                          str          str          free text              (empty)  ← CHECK
  First Name                     str          str          free text              (empty)  ← CHECK
  Last Name                      str          str          free text              (empty)  ← CHECK
  Email                          str          str          free text              free text
  Phone                          str          str          free text              (empty)  ← CHECK
  Birthdate                      str          str          free text              (empty)  ← CHECK
  Address

,Membership Number #,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,Membership,Gender,Note
3,11929,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €",ESPN&IPNA lower-middle income - online journal,NaN,



  IPNA Masterliste.csv
  Columns missing : —
  Extra columns   : —
  Column order OK : True

  Column                         orig dtype   new dtype    orig pattern           new pattern
  ---------------------------------------------------------------------------------------------------------
  Titel                          str          str          free text              (empty)  ← CHECK
  First Name                     str          str          free text              (empty)  ← CHECK
  Last Name                      str          str          free text              (empty)  ← CHECK
  Email                          str          str          free text              free text
  Phone                          str          str          free text              (empty)  ← CHECK
  Birthdate                      str          str          free text              (empty)  ← CHECK
  Address                        str          str          free text              free text
  City                   

,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,IPNA amount,Membership,Gender,Note
4,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €","52,00 €",ESPN&IPNA lower-middle income - online journal,NaN,
